# BO Forge replicate-aware campaign

This notebook demonstrates the replicate-aware `CampaignSession` workflow for explicit replicate rows, group-mean model fitting, and replicate-derived observation variance.

The design reference is `/Users/liangze/Desktop/from-pytorch-to-bayesian-optimisation/part_6/tutorial_01_noisy_and_replication_aware_bo.ipynb`, especially empirical replicate variance, noisy GP fitting, repeat-vs-explore decisions, and replicate-aware diagnostics.

## 1. Setup

The seed log contains manual replicate rows. BO Forge keeps every raw row in the CSV, fits the model on replicate-group means, and passes empirical group-mean variance to BoTorch as `train_Yvar` once at least one group has repeated observations.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.models import dataframe_to_training_tensors
from bo_forge.session import CampaignSession

In [ ]:
config_path = PROJECT_ROOT / "configs" / "08_replicate_aware_logei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "08_replicate_aware_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "08_replicate_aware_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "08_replicate_aware_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "08_replicate_aware_report.txt"
progress_path = PROJECT_ROOT / "reports" / "08_replicate_aware_progress.pdf"
replicate_plot_path = PROJECT_ROOT / "reports" / "08_replicate_aware_replicates.pdf"
TARGET_REPLICATE_GROUPS = 15

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path, working_log_path)

## 2. Inspect raw rows and replicate groups

In [ ]:
campaign.validate()

display(campaign.summary())
display(campaign.replicate_summary())
display(campaign.best_observation())
display(campaign.best_replicate_group())
display(campaign.next_action())

training_tensors = dataframe_to_training_tensors(campaign.config, campaign.observed_data())
if training_tensors.train_yvar is not None:
    display(
        pd.DataFrame(
            training_tensors.train_yvar.numpy(),
            columns=campaign.config.objective_names,
        ).rename_axis("training_group")
    )

campaign.df

## 3. Synthetic noisy objective

This simulator has a smooth optimum plus deterministic pseudo-noise. The seed log already shows why raw best and mean best can differ, and why an uncertain best design may deserve another repeat before exploring elsewhere.

In [ ]:
def simulate_activity(row):
    ratio = float(row["precursor_ratio"])
    temperature = float(row["annealing_temperature"])
    temp_scaled = (temperature - 300.0) / 500.0

    peak = np.exp(
        -0.5
        * (
            ((ratio - 0.58) / 0.16) ** 2
            + ((temp_scaled - 0.62) / 0.18) ** 2
        )
    )
    structured_noise = 0.05 * np.sin(13.0 * ratio + 0.02 * temperature)
    return float(1.0 + peak + structured_noise)

## 4. Complete the initial design

The initial design size counts replicate groups, not raw replicate rows. Initial generated suggestions remain exploration suggestions and start a new replicate group.

In [ ]:
suggestions = campaign.suggest_next(batch_size=1)
suggestions.to_csv(latest_suggestion_path, index=False)

display(suggestions)
display(campaign.suggestion_quality(suggestions))

campaign.append_suggestions(suggestions)
suggested_row = suggestions.iloc[0]
observed_value = simulate_activity(suggested_row)
campaign.mark_observed(
    row_id=str(suggested_row["row_id"]),
    objective_value=observed_value,
)

display(pd.DataFrame([{"observed_value": observed_value}]))
display(campaign.replicate_summary())
display(campaign.summary())

## 5. Run model-based BO rounds

After enough replicate groups exist, BO Forge fits the GP on group means and uses replicate-derived `train_Yvar` when empirical replicate variance is available. With the default `uncertain_best` policy, a model-based suggestion may repeat the current best group if it is still uncertain or under-replicated; use `suggestion_policy: new_only` for exploration-only suggestions. This notebook demonstrates the workflow rather than qLogNEI, qLogNEHVI, learned heteroskedastic noise modelling, or multi-objective repeat policy.

In [ ]:
bo_rounds = []

while len(campaign.replicate_summary()) < TARGET_REPLICATE_GROUPS:
    bo_suggestions = campaign.suggest_next(batch_size=1)
    bo_suggestions.to_csv(latest_suggestion_path, index=False)

    display(campaign.suggestion_quality(bo_suggestions))
    bo_rounds.append(
        pd.DataFrame(
            [
                {
                    "bo_round": len(bo_rounds) + 1,
                    "row_id": bo_suggestions.loc[0, "row_id"],
                    "precursor_ratio": bo_suggestions.loc[0, "precursor_ratio"],
                    "annealing_temperature": bo_suggestions.loc[0, "annealing_temperature"],
                    "predicted_mean": bo_suggestions.loc[0, "predicted_mean"],
                    "predicted_std": bo_suggestions.loc[0, "predicted_std"],
                    "acquisition": bo_suggestions.loc[0, "acquisition"],
                }
            ]
        )
    )

    campaign.append_suggestions(bo_suggestions)
    suggested_row = bo_suggestions.iloc[0]
    campaign.mark_observed(
        row_id=str(suggested_row["row_id"]),
        objective_value=simulate_activity(suggested_row),
    )
    campaign.reload()

assert len(campaign.replicate_summary()) == TARGET_REPLICATE_GROUPS

display(pd.concat(bo_rounds, ignore_index=True))
display(campaign.replicate_summary())
display(campaign.summary())
display(campaign.next_action())

## 6. Report and plots

In [ ]:
report = campaign.report()
display(report["summary"])
display(report["best_observation"])
display(report["best_replicate_group"])

campaign.export_report(report_path)
campaign.plot_progress(save_path=progress_path)
campaign.plot_replicates(save_path=replicate_plot_path)

pd.DataFrame(
    [
        {"artifact": "report", "path": str(report_path)},
        {"artifact": "progress plot", "path": str(progress_path)},
        {"artifact": "replicate plot", "path": str(replicate_plot_path)},
    ]
)